# 🔍 Notebook 5: Retrieval
### Finding the right chunks — the 'R' in RAG

**WAT Reference:** `workflows/05_retrieve_chunks.md` | `tools/retrieve.py`

---

## What is Retrieval?

You have thousands of chunks, each with its embedding (1,536 numbers). When you ask a question:

1. Your question is also converted to 1,536 numbers
2. We compare your question's numbers to every chunk's numbers
3. The chunks with the most similar numbers = the most relevant content
4. We return the top-5 most similar chunks

This is **semantic search** — it finds relevant content even if the exact words differ.

```
Question: "What is competitive advantage?"
   ↓  embed
[0.041, -0.318, 0.204, ...]  ← question vector
   ↓  compare to all chunks
Score 0.91: "Sustainable competitive advantage arises from..." (Porter.pdf)
Score 0.87: "Competitive moats include brand, network effects..." (Strategy Notes.docx)
Score 0.83: "Value chain analysis identifies sources of advantage..." (BCG Case.pptx)
```

## What You'll Do
1. Load the vector store from Notebook 4
2. Embed a question and see its vector
3. Compute similarity against all chunks (manual, step by step)
4. Build an interactive search — try different questions
5. Use subject filtering to narrow results

**Time needed:** ~15 minutes | **Cost:** ~$0.001 (just embedding your questions)

In [ ]:
#@title ⚙️ Step 1 — Setup: install, connect Drive, load API key  { display-mode: "form" }
!pip install openai -q

import os, json, math
import openai

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")

PROJECT_ROOT    = '/content/drive/MyDrive/MBA_RAG'
VECTOR_STORE    = os.path.join(PROJECT_ROOT, '.tmp/vector_store.json')
EMBEDDING_MODEL = 'text-embedding-3-small'

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
if not OPENAI_API_KEY and IN_COLAB:
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
        if OPENAI_API_KEY:
            os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    except Exception:
        pass

if OPENAI_API_KEY:
    print(f"✅ API key loaded. Starts with: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  API key not found. Open 🔑 Secrets → Add OPENAI_API_KEY → re-run.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

if IN_COLAB and os.path.exists(VECTOR_STORE):
    with open(VECTOR_STORE, 'r', encoding='utf-8') as f:
        chunks = json.load(f)
    subjects = sorted(set(c.get('subject', '') for c in chunks))
    print(f"✅ Loaded {len(chunks):,} chunks")
    print(f"   Subjects: {subjects}")
elif IN_COLAB:
    print(f"⚠️  vector_store.json not found. Run Notebooks 2 → 3 → 4 first.")
    chunks, subjects = [], []
else:
    chunks, subjects = [], []

---
## Step 1: Embed Your Question

Before we can compare a question to chunks, we need to convert the question into the same format — a 1,536-number vector using the **same embedding model**.

In [ ]:
#@title 🔧 Background — Helper functions (run once)  { display-mode: "form" }
def embed_text(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding

def cosine_similarity(vec_a, vec_b):
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

print("✅ Functions ready.")

---
## Step 2: Cosine Similarity — Step by Step

We compare the question vector to every chunk vector and compute a score.
Let's do the first few comparisons manually so you see exactly what happens.

In [ ]:
#@title 📐 Step 2 — See how similarity scores work  { display-mode: "form" }
#@markdown Type a question below and see the similarity scores against your first 5 chunks.
demo_question = "What is competitive strategy?" #@param {type:"string"}

question_vector = embed_text(demo_question)
print(f'Question: "{demo_question}"')
print(f"Converted to: {len(question_vector)} numbers")
print()
print("Comparing against first 5 chunks:")
print("─" * 70)

for i, chunk in enumerate(chunks[:5]):
    score   = cosine_similarity(question_vector, chunk['embedding'])
    preview = chunk['text'][:80].replace('\n', ' ')
    print(f"\nChunk {i} | Score: {score:.4f}")
    print(f"Source:  {chunk['filename']}")
    print(f"Text:    \"{preview}...\"")

print()
print("We do this for ALL chunks, then return the top 5.")

In [ ]:
#@title 🔧 Background — Full retrieval function (run once)  { display-mode: "form" }
def retrieve(query, subject_filter=None, top_k=5):
    query_vector = embed_text(query)
    pool = chunks
    if subject_filter:
        pool = [c for c in chunks if c.get('subject','').lower() == subject_filter.lower()]
        if not pool:
            print(f"⚠️  No chunks for subject '{subject_filter}'. Using all subjects.")
            pool = chunks
    scored = [
        {
            'score':    round(cosine_similarity(query_vector, c['embedding']), 4),
            'filename': c['filename'],
            'subject':  c.get('subject', ''),
            'text':     c['text'],
            'chunk_id': c['chunk_id'],
        }
        for c in pool
    ]
    scored.sort(key=lambda x: x['score'], reverse=True)
    return scored[:top_k]

print("✅ Retrieval function ready.")

---
## Step 3: Subject Filtering — Precision Retrieval

When you know which subject area a question belongs to, filtering first reduces noise significantly.

In [ ]:
#@title 🔍 Step 3 — Search all subjects vs. filtered by subject  { display-mode: "form" }
#@markdown Change the question and subject below to search your MBA files.
finance_question = "How is WACC used in valuation?" #@param {type:"string"}
subject_filter   = "Finance" #@param {type:"string"}

print(f'Question: "{finance_question}"')
print()
print("─" * 60)
print("WITHOUT subject filter (all subjects):")
print("─" * 60)
results_all = retrieve(finance_question)
for i, r in enumerate(results_all, 1):
    print(f"[{i}] {r['score']} | {r['filename'][:45]} ({r['subject']})")

print()
print(f"─" * 60)
print(f"WITH subject='{subject_filter}' filter:")
print("─" * 60)
results_filtered = retrieve(finance_question, subject_filter=subject_filter)
for i, r in enumerate(results_filtered, 1):
    print(f"[{i}] {r['score']} | {r['filename'][:45]} ({r['subject']})")

print()
print(f"Available subjects: {subjects}")

---
## Step 4: Interactive Search — Try Your Own Questions

Use the cell below to search your MBA knowledge base. Change the question and subject as needed.

In [ ]:
#@title 💬 Step 4 — Search your MBA files  { display-mode: "form" }
#@markdown Type any question below. Optionally set a subject to narrow the search.
MY_QUESTION = "What frameworks did I study for strategic analysis?" #@param {type:"string"}
SUBJECT     = "" #@param {type:"string"}
TOP_K       = 5  #@param {type:"integer", min:1, max:10}

subject_arg = SUBJECT if SUBJECT.strip() else None
print(f'Searching: "{MY_QUESTION}"')
if subject_arg:
    print(f"Subject filter: {subject_arg}")
print()

my_results = retrieve(MY_QUESTION, subject_filter=subject_arg, top_k=TOP_K)

print(f"TOP {len(my_results)} RESULTS")
print("=" * 70)
for i, r in enumerate(my_results, 1):
    print(f"\n[{i}] Score: {r['score']}")
    print(f"     File:    {r['filename']}  ({r['subject']})")
    print(f"     Content: {r['text'][:250].replace(chr(10), ' ')}...")

print()
print("Score guide:  0.8+ = very relevant  |  0.6–0.8 = relevant  |  <0.6 = less relevant")

---
## ✅ Notebook 5 Complete!

### What you learned:
1. **Retrieval** = embed question → compute cosine similarity → return top-K chunks
2. **Semantic search** finds relevant content even with different words
3. **Subject filtering** improves precision when you know the domain
4. **Similarity scores** tell you how confident the retrieval is

### Key insight:
> The retriever doesn't understand the question. It just finds the chunks whose numbers are closest to the question's numbers. The understanding happens in the next step — when GPT-4 reads those chunks.

---
### Next: Notebook 6 — Full Pipeline

We connect everything: question → retrieve → GPT-4 → answer with sources. Then we'll also show how to switch from the JSON file to ChromaDB for the full 1,250-file corpus.

**Update README.md — check off Notebook 5 ✅**